[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mbanuelos/grad_math_modeling/blob/main/Lectures/Module2_DataSimilarity/05_Similarity_SVD.ipynb#copy=true)


# Similarity and Singular Value Decomposition

## Learning objectives

By the end of this lesson, you should be able to:

- Calculate and interpret cosine similarity between feature vectors.
- Explain how SVD factors a matrix into directions and singular values.
- Relate a low-rank approximation to compression and latent structure.

**Student Learning Outcome (SLO 6):**
> Compare objects with cosine similarity and use a low-rank SVD approximation to describe structure in a matrix.


## Part 1 — Ratings as an item–user matrix

A recommender can represent ratings in a matrix: each row is a movie, each column is a user, and an entry is that user's rating of that movie. A row is therefore a movie's rating profile. Real matrices are mostly missing because most people have not rated most movies; this small, fully observed example lets us focus first on the geometry of the rows.

In the next lesson, we will return to the important question of how to handle those missing ratings.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

item_user = pd.DataFrame(
    [[5, 4, 5, 4],
     [4, 5, 4, 5],
     [1, 2, 1, 2],
     [2, 1, 2, 1]],
    index=["Galaxy Quest", "Star Voyage", "Quiet Portrait", "Storm Warning"],
    columns=["User A", "User B", "User C", "User D"],
)
item_user


### ✏️ Written response 1

What does one row represent? What does one column represent? If a real user had not rated a movie, what would be missing from this table?

> **YOUR ANSWER:**


## Part 2 — Direct item neighborhoods with cosine similarity

To find movies with similar audiences, compare their rating-profile rows. For nonzero vectors $a$ and $b$, cosine similarity is

$$\cos(\theta) = \frac{a\cdot b}{\|a\|\|b\|}.$$

It is high when two rows point in similar directions. Before comparing ratings, subtract each user's average rating so the vectors describe above- and below-usual preferences rather than whether someone generally gives high scores. In this complete toy matrix, that centering is easy to see; the next lesson applies the same idea to observed MovieLens ratings.


In [ ]:
centered_profiles = item_user.sub(item_user.mean(axis=0), axis=1)
item_similarity = pd.DataFrame(
    cosine_similarity(centered_profiles),
    index=item_user.index,
    columns=item_user.index,
)
item_similarity.round(2)


### Checkpoint

Which movie is the closest neighbor of `Galaxy Quest` besides itself? Describe the rating-pattern evidence. Why might a direct cosine-neighbor method struggle if two movies have very few users in common?

> **YOUR ANSWER:**


## Part 3 — From local neighbors to global structure

Cosine similarity compares two rows at a time. That is useful for neighborhoods, but a recommender with many movies and users needs to recognize patterns shared across the *whole* item–user matrix. If several movies tend to be liked by the same groups of users, the matrix may have lower-dimensional structure even when it has many rows and columns.

A low-rank approximation summarizes those shared patterns using a few latent directions. It is not another similarity metric: it is a way to model the entire matrix before comparing or estimating entries.


In [ ]:
M = centered_profiles.to_numpy(dtype=float)
U, singular_values, Vt = np.linalg.svd(M, full_matrices=False)
print("singular values:", singular_values.round(3))

def rank_k_approximation(matrix, k):
    U, S, Vt = np.linalg.svd(matrix, full_matrices=False)
    return (U[:, :k] * S[:k]) @ Vt[:k, :]

rank_1 = pd.DataFrame(
    rank_k_approximation(M, 1), index=item_user.index, columns=item_user.columns
)
rank_1.round(1)


### Partner check

Compare the original ratings matrix with its rank-1 approximation. Which broad preference pattern remains? Which movie-specific or user-specific details are no longer exact?

> **YOUR ANSWER:**


## Part 4 — SVD and low-rank approximation

The singular value decomposition factors an $m\times n$ matrix as

$$M = U\Sigma V^T.$$

For an item–user matrix, columns of $U$ describe latent item patterns, rows of $V^T$ describe latent user patterns, and the diagonal values in $\Sigma$ rank those patterns by strength. Keeping the largest $k$ singular values gives the best rank-$k$ approximation in Frobenius norm: a compact summary that preserves as much of the matrix as possible under that measure.


In [ ]:
for k in range(1, min(M.shape) + 1):
    approximation = rank_k_approximation(M, k)
    error = np.linalg.norm(M - approximation, ord="fro")
    print(f"rank {k}: reconstruction error = {error:.3f}")

fig, axes = plt.subplots(1, 3, figsize=(11, 3))
for ax, matrix, title in zip(
    axes, [M, rank_k_approximation(M, 1), rank_k_approximation(M, 2)],
    ["observed profiles", "rank 1 summary", "rank 2 summary"],
):
    image = ax.imshow(matrix, cmap="coolwarm", vmin=-2, vmax=2)
    ax.set_title(title)
    ax.set_xticks(range(len(item_user.columns)), item_user.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(item_user.index)), item_user.index)
fig.colorbar(image, ax=axes, shrink=0.8, label="centered rating")
plt.tight_layout()
plt.show()


### Written response 2

How does $k$ affect reconstruction error? Why is a low-rank summary useful when rating profiles overlap only partially? What useful movie-specific detail could it remove?

> **YOUR ANSWER:**


## Part 5 — Preparing for real recommendation data

This example is intentionally complete. Real item–user matrices are sparse: an empty cell means “not rated,” not a low rating. The next lesson uses MovieLens data, centers each user's observed ratings before computing cosine item neighborhoods, and compares that direct approach with the latent-factor idea introduced here.

The distinction to carry forward is:

- **Neighborhood recommendation:** compare observed item rows directly with a similarity measure.
- **Latent-factor recommendation:** use a low-rank model to describe shared item and user patterns, then use that structure to help estimate preferences.


### Exit prompt

A new movie has been rated by only three people. Why might direct cosine neighbors be unreliable? How could a latent-factor approach help, and what information would still be unavailable for a completely unrated movie?

> **YOUR ANSWER:**


## Lesson summary

An item–user matrix turns rating histories into vectors. Cosine similarity builds direct item neighborhoods from those vectors. SVD finds low-rank, latent structure across the whole matrix. Both methods depend on how ratings are represented, and real recommendation data requires careful treatment of missing feedback, evaluation, and user impact.


### Looking ahead

The recommendation-systems lesson will use a much larger, sparse MovieLens matrix. It will implement centered cosine item neighborhoods and examine missingness, held-out evaluation, cold starts, and fairness; it will treat latent factors as a useful direction rather than a finished production solution.
